# 01 — Data Cleaning & Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/Ecommerce_Delivery_Analytics_New.csv')
print(df.shape)
df.head()


(100000, 11)


,Order ID,Customer ID,Platform,Order Date & Time,Delivery Time (Minutes),Product Category,Order Value (INR),Customer Feedback,Service Rating,Delivery Delay,Refund Requested
0,ORD000001,CUST2824,JioMart,19:29.5,30,Fruits & Vegetables,382,"Fast delivery, great service!",5,No,No
1,ORD000002,CUST1409,Blinkit,54:29.5,16,Dairy,279,Quick and reliable!,5,No,No
2,ORD000003,CUST5506,JioMart,21:29.5,25,Beverages,599,Items missing from order.,2,No,Yes
3,ORD000004,CUST5012,JioMart,19:29.5,42,Beverages,946,Items missing from order.,2,Yes,Yes
4,ORD000005,CUST4657,Blinkit,49:29.5,30,Beverages,334,"Fast delivery, great service!",5,No,No


In [2]:
# basic checks
print("Missing values:")
print(df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
print("\nDtypes:")
print(df.dtypes)


Missing values:
Order ID                   0
Customer ID                0
Platform                   0
Order Date & Time          0
Delivery Time (Minutes)    0
Product Category           0
Order Value (INR)          0
Customer Feedback          0
Service Rating             0
Delivery Delay             0
Refund Requested           0
dtype: int64

Duplicates: 0

Dtypes:
Order ID                     str
Customer ID                  str
Platform                     str
Order Date & Time            str
Delivery Time (Minutes)    int64
Product Category             str
Order Value (INR)          int64
Customer Feedback            str
Service Rating             int64
Delivery Delay               str
Refund Requested             str
dtype: object


## Feature Engineering

In [2]:
# extract hour from time column (format is HH:MM.S)
df['Hour'] = df['Order Date & Time'].str.split(':').str[0].astype(int)

# binary flags — makes everything downstream cleaner
df['Delay_Flag']  = (df['Delivery Delay']   == 'Yes').astype(int)
df['Refund_Flag'] = (df['Refund Requested'] == 'Yes').astype(int)

print("Hour sample:", df['Hour'].head().values)
print("Delay rate: ", round(df['Delay_Flag'].mean()*100, 1), "%")
print("Refund rate:", round(df['Refund_Flag'].mean()*100, 1), "%")


Hour sample: [19 54 21 19 49]
Delay rate:  13.7 %
Refund rate: 45.8 %


In [3]:
# delivery speed tiers — based on quick-commerce SLAs
def get_tier(mins):
    if mins <= 20:   return 'Express'
    elif mins <= 35: return 'On-Time'
    elif mins <= 50: return 'Slow'
    else:            return 'Very Late'

df['Delivery_Tier'] = df['Delivery Time (Minutes)'].apply(get_tier)

# order value segments
df['Order_Segment'] = pd.cut(
    df['Order Value (INR)'],
    bins=[0, 200, 500, 1000, 2000],
    labels=['Low', 'Medium', 'High', 'Premium']
)

# hour buckets for peak analysis
def get_bucket(h):
    if   7 <= h <= 10: return 'Morning'
    elif 12 <= h <= 14: return 'Lunch'
    elif 18 <= h <= 22: return 'Evening'
    else:               return 'Off-Peak'

df['Hour_Bucket'] = df['Hour'].apply(get_bucket)

print(df['Delivery_Tier'].value_counts())


Delivery_Tier
On-Time      53992
Slow         25782
Express      18473
Very Late     1753
Name: count, dtype: int64


In [4]:
# save for all downstream notebooks
df.to_csv('../data/cleaned_data.csv', index=False)
print("Saved. Shape:", df.shape)
df.sample(3)


Saved. Shape: (100000, 17)


,Order ID,Customer ID,Platform,Order Date & Time,Delivery Time (Minutes),Product Category,Order Value (INR),Customer Feedback,Service Rating,Delivery Delay,Refund Requested,Hour,Delay_Flag,Refund_Flag,Delivery_Tier,Order_Segment,Hour_Bucket
89055,ORD089056,CUST1675,Blinkit,42:29.5,34,Fruits & Vegetables,587,"Very late delivery, not happy.",2,No,Yes,42,0,1,On-Time,High,Off-Peak
56108,ORD056109,CUST8543,Swiggy Instamart,40:29.5,40,Personal Care,1032,Quick and reliable!,5,No,No,40,0,0,Slow,Premium,Off-Peak
96409,ORD096410,CUST9136,Swiggy Instamart,05:29.5,26,Beverages,349,Packaging could be better.,3,No,No,5,0,0,On-Time,Medium,Off-Peak
